# RAG Mastery: Unified Project Playground

An interactive end-to-end sandbox integrating multi-format data ingestion, document chunking, remote vector embeddings, FAISS indexing, semantic search, and 2D semantic visualization.

> For comprehensive theoretical explanations, design patterns, and pipeline deep-dives, see the [Playground Documentation](README.md).

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np

# Guard against blocked native libraries under Windows Smart App Control
for blocked_mod in ['torch', 'sentence_transformers', 'spacy']:
    sys.modules[blocked_mod] = None

# Configure paths
current_dir = Path.cwd()
project_root = current_dir if (current_dir / 'data').exists() else current_dir.parent
indexing_dir = project_root / '02-indexing'

for path_to_add in [project_root, indexing_dir]:
    if str(path_to_add) not in sys.path:
        sys.path.insert(0, str(path_to_add))

# Load environment variables
from dotenv import load_dotenv
load_dotenv(project_root / '.env')

# Verify remote embedding service readiness
from huggingface_api import HuggingFaceAPIEmbeddings

hf_token = os.getenv('HF_TOKEN')
print(f"Project root: {project_root}")
print(f"HF_TOKEN detected: {'Yes (Valid)' if hf_token else 'No (Set HF_TOKEN in .env)'}")


## 1. Multi-Format Data Ingestion
Load documents from multiple file formats (`.txt`, `.pdf`, `.docx`, `.csv`) located in the `data/` directory into LangChain document objects.

In [ ]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader, Docx2txtLoader, CSVLoader
from langchain_core.documents import Document

def load_sample_document(file_path: Path) -> list[Document]:
    """Load a document based on its file extension."""
    ext = file_path.suffix.lower()
    try:
        if ext == '.txt':
            return TextLoader(str(file_path), encoding='utf-8').load()
        elif ext == '.pdf':
            return PyPDFLoader(str(file_path)).load()
        elif ext == '.docx':
            return Docx2txtLoader(str(file_path)).load()
        elif ext == '.csv':
            return CSVLoader(str(file_path)).load()
    except Exception as e:
        print(f"Could not load {file_path.name}: {e}")
    return []

# Ingest sample files across different formats
sample_files = [
    project_root / 'data' / 'txt' / 'artificial_intelligence.txt',
    project_root / 'data' / 'pdf' / 'sample_paper.pdf',
    project_root / 'data' / 'docx' / 'proposal.docx',
    project_root / 'data' / 'tabular' / 'products.csv',
]

all_documents = []
for file_p in sample_files:
    if file_p.exists():
        loaded = load_sample_document(file_p)
        all_documents.extend(loaded)
        char_count = sum(len(d.page_content) for d in loaded)
        print(f"[+] Loaded {len(loaded):2d} doc(s) from: {file_p.name} ({char_count:,} chars)")

print(f"\nTotal loaded documents/pages: {len(all_documents)}")


## 2. Text Splitting & Chunking
Split documents into semantically coherent chunks using `RecursiveCharacterTextSplitter` with configurable chunk size and overlap.

In [ ]:
from langchain_text_splitters.character import RecursiveCharacterTextSplitter

# Initialize text splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=50,
    separators=['\n\n', '\n', '. ', ' ', '']
)

# Generate chunks
chunks = splitter.split_documents(all_documents)
print(f"Split {len(all_documents)} document(s) into {len(chunks)} chunks.")

# Display sample chunk details
print('\n--- Sample Chunk Preview ---')
sample_chunk = chunks[0]
source_name = Path(sample_chunk.metadata.get('source', 'unknown')).name
print(f"Source: {source_name}")
print(f"Length: {len(sample_chunk.page_content)} characters")
snippet = sample_chunk.page_content.strip().replace('\n', ' ')
print(f"Content: {snippet[:180]}...")


## 3. Remote Vector Embeddings
Transform text into 384-dimensional dense vectors using the remote Hugging Face Inference API (`sentence-transformers/all-MiniLM-L6-v2`) with zero local PyTorch dependencies.

In [ ]:
# Initialize remote embedding model
embeddings = HuggingFaceAPIEmbeddings(
    model='sentence-transformers/all-MiniLM-L6-v2',
    normalize_embeddings=True,
    expected_dimension=384
)

# Test query embedding
test_text = 'What is artificial intelligence and machine learning?'
query_vector = embeddings.embed_query(test_text)

vec_np = np.array(query_vector)
print(f"Input text: '{test_text}'")
print(f"Vector dimension: {len(query_vector)}")
print(f"L2 Norm: {np.linalg.norm(vec_np):.4f} (normalized: {np.isclose(np.linalg.norm(vec_np), 1.0)})")
print(f"First 5 coordinates: {np.round(vec_np[:5], 4)}")


## 4. Semantic Similarity Matrix
Compute pairwise cosine similarity between sample domain concepts to evaluate semantic proximity.

In [ ]:
concepts = [
    'Artificial intelligence and deep neural networks',
    'Machine learning algorithms and predictive models',
    'Financial investment, stock market, and bonds',
    'Banking, corporate finance, and interest rates',
    'Healthy nutrition, organic diet, and vitamins',
    'Cardiovascular exercise and physical training',
]

concept_vectors = np.array(embeddings.embed_documents(concepts))

# Pairwise cosine similarity matrix (dot product of L2-normalized vectors)
similarity_matrix = np.dot(concept_vectors, concept_vectors.T)

# Print formatted similarity table
short_labels = ['AI/DL', 'ML/Models', 'Stocks', 'Banking', 'Diet', 'Fitness']
header = f"{'Concept':<10}" + ''.join(f"{lbl:>10}" for lbl in short_labels)
print(header)
print('-' * len(header))
for i, row in enumerate(similarity_matrix):
    row_str = f"{short_labels[i]:<10}" + ''.join(f"{val:>10.3f}" for val in row)
    print(row_str)


## 5. FAISS Vector Store Indexing
Index document chunks into an in-memory FAISS index for high-performance approximate nearest neighbor retrieval.

In [ ]:
from langchain_community.vectorstores import FAISS

# Index the first 25 chunks for interactive experimentation
selected_chunks = chunks[:25]
vectorstore = FAISS.from_documents(selected_chunks, embeddings)

print(f"Indexed {vectorstore.index.ntotal} chunks into FAISS vector store.")
print(f"Vector dimension: {vectorstore.index.d}")


## 6. End-to-End Retrieval & Relevance Scoring
Execute natural language queries against the FAISS index and inspect retrieved chunks with converted cosine similarity scores.

In [ ]:
query = 'What is machine learning and how is it used?'
top_k = 3

print(f"Query: '{query}'\n")

# Similarity search with FAISS L2 distance
results_with_scores = vectorstore.similarity_search_with_score(query, k=top_k)

for rank, (doc, l2_distance) in enumerate(results_with_scores, 1):
    # Convert FAISS L2 distance to cosine similarity for normalized vectors: sim = 1 - (d^2 / 2)
    cosine_sim = 1.0 - (l2_distance ** 2) / 2.0
    source_name = Path(doc.metadata.get('source', 'Unknown')).name
    
    print(f"Rank {rank}:")
    print(f"  Source:     {source_name}")
    print(f"  Similarity: {cosine_sim:.4f} (L2 Distance: {l2_distance:.4f})")
    snippet = doc.page_content.strip().replace('\n', ' ')
    print(f"  Snippet:    {snippet[:140]}...")
    print('-' * 60)

# Verify LangChain Retriever interface
retriever = vectorstore.as_retriever(search_kwargs={'k': 2})
retrieved_docs = retriever.invoke(query)
print(f"\nRetriever interface test passed: {len(retrieved_docs)} docs returned.")


## 7. 2D Semantic Space Visualization (PCA)
Project high-dimensional embeddings into a 2D plane using Principal Component Analysis (SVD) to inspect domain clustering visually.

In [ ]:
import matplotlib.pyplot as plt

# Domain-specific sentences
labeled_data = [
    ('Tech', 'Deep learning models require GPUs for training'),
    ('Tech', 'Natural language processing analyzes human text'),
    ('Tech', 'Computer vision classifies objects in images'),
    ('Finance', 'The central bank decided to lower interest rates'),
    ('Finance', 'Stock market portfolios need risk diversification'),
    ('Finance', 'Corporate financial statements show revenue and debt'),
    ('Health', 'Cardiovascular exercise strengthens the heart'),
    ('Health', 'A balanced diet provides essential vitamins and minerals'),
    ('Health', 'Clinical medical trials test new drug treatments'),
]

labels = [item[0] for item in labeled_data]
texts = [item[1] for item in labeled_data]

# Embed texts
vecs = np.array(embeddings.embed_documents(texts))

# Pure NumPy SVD projection to 2D (PCA)
vecs_centered = vecs - np.mean(vecs, axis=0)
u, s, vt = np.linalg.svd(vecs_centered, full_matrices=False)
coords_2d = u[:, :2] * s[:2]

# Plot
plt.figure(figsize=(9, 6))
color_map = {'Tech': '#2563EB', 'Finance': '#16A34A', 'Health': '#DC2626'}

for category in ['Tech', 'Finance', 'Health']:
    idx = [i for i, lbl in enumerate(labels) if lbl == category]
    plt.scatter(
        coords_2d[idx, 0], coords_2d[idx, 1],
        label=category, color=color_map[category], s=120, edgecolors='black', linewidths=1.2
    )

for i, txt in enumerate(texts):
    short_txt = (txt[:28] + '..') if len(txt) > 28 else txt
    plt.annotate(
        short_txt,
        (coords_2d[i, 0], coords_2d[i, 1]),
        textcoords='offset points',
        xytext=(6, 6),
        fontsize=9,
        alpha=0.85
    )

plt.title('2D Semantic Embedding Space (PCA Projection)', fontsize=13, fontweight='bold', pad=12)
plt.xlabel('Principal Component 1', fontsize=10)
plt.ylabel('Principal Component 2', fontsize=10)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(frameon=True)
plt.tight_layout()
plt.show()
